# A2 · K-fold, stratified K-fold and `cross_val_score` from scratch

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Implement the generation of folds (identical to scikit-learn's `KFold`), a stratified version and a
`my_cross_val_score` function, and verify that with the same folds they produce the same scores.

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns and the target `defaulted` (about 19% positives). Model: logistic
regression with a scaler in a pipeline.

## Educational purpose

Understand what cross-validation really does (partition, train from scratch on each training set, evaluate on the rest) and how
stratification keeps the class proportions.

## Task

1. Write `kfold_indices(n, k, seed)`: use `np.random.RandomState(seed).shuffle` on `np.arange(n)`; the folds have size `n//k` and the first `n % k` have one more. Return the list of pairs `(train_idx, val_idx)` with **sorted** indices. It must match `KFold(k, shuffle=True, random_state=seed)`.
2. Write `strat_indices(y, k, seed)`: for every class shuffle its indices (`np.random.RandomState(seed)`) and assign them to the folds in turn (`i % k`). Return the list of pairs `(train_idx, val_idx)` sorted. In every fold the number of elements of each class must not differ by more than 1 from that of the other folds.
3. Write `my_cross_val_score(model, X, y, folds)`: for every pair clone the model with `sklearn.base.clone`, train it on the training set, compute the AUC (`roc_auc_score` with `predict_proba`) on the validation fold and return the array of scores.
4. Compute `my_scores = my_cross_val_score(model, X, y, kfold_indices(len(y), 5, 0))` and compare them with scikit-learn's `cross_val_score` passing it the **same** folds as `cv`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y = df["defaulted"].to_numpy()
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

## Your work

In [ ]:
def kfold_indices(n, k, seed):
    ...


def strat_indices(y, k, seed):
    ...


def my_cross_val_score(model, X, y, folds):
    ...

In [ ]:
folds = kfold_indices(len(y), 5, 0)
my_scores = ...
sk_scores = cross_val_score(model, X, y, cv=folds, scoring="roc_auc")
print(my_scores.round(4), sk_scores.round(4))

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Sizes: `size = np.full(k, n // k); size[: n % k] += 1`. With `pos = np.cumsum(np.r_[0, size])` fold i is `idx[pos[i]:pos[i+1]]` (then sort it). Training: `np.setdiff1d(np.arange(n), val)`. Stratification: `fold_of = np.empty(n, int)`, for every class `ind = np.flatnonzero(y == c); r.shuffle(ind); fold_of[ind] = np.arange(len(ind)) % k`.'):
    ref = list(KFold(5, shuffle=True, random_state=0).split(np.arange(len(y))))
    check(len(folds) == 5 and all(np.array_equal(a[1], b[1]) and np.array_equal(a[0], b[0]) for a, b in zip(folds, ref)),
          "kfold_indices matches scikit-learn's KFold(shuffle=True)", "same shuffle (RandomState(seed)), folds of size n//k with the remainder on the first ones")
    everyone = np.sort(np.concatenate([v for _, v in folds]))
    check(np.array_equal(everyone, np.arange(len(y))), "the validation folds partition the data", "every row in exactly one validation fold")
    sf = strat_indices(y, 5, 0)
    counts = np.array([[(y[v] == c).sum() for c in (0, 1)] for _, v in sf])
    check(np.all(counts.max(axis=0) - counts.min(axis=0) <= 1), "stratification balances every class across the folds (difference at most 1)",
          "for every class, the counts in the folds must differ by at most 1")
    check(all(len(np.intersect1d(t, v)) == 0 and len(t) + len(v) == len(y) for t, v in sf), "training and validation are disjoint and complete",
          "every pair must partition all the rows")
    check(close(my_scores, sk_scores), "my_cross_val_score matches cross_val_score on the same folds",
          "clone the model at every fold, train on the training part, AUC on the validation fold")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_kfold_and_cross_val_score_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.